In [30]:
# ── 셀 1 │ 설정 ──
import os, re, json, hashlib, datetime, collections
import numpy as np, pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog('rdApp.*')

NEW       = 'C:/Users/iisl/Desktop/0_Molecualar_Informatics/2026_final/2026_new/'
CLIN_RAW  = NEW + 'data/raw/sider_dictrank/'
CLIN_OUT  = NEW + 'data/processed/clinical/'
CM_OUT    = NEW + 'data/processed/crediblemeds/'
HERG_POOL = NEW + 'data/processed/herg_karim/herg_karim_pool.csv'
LABEL     = 'Cardiotox (with SIDER inactives)'
os.makedirs(CLIN_OUT, exist_ok=True)

def pick(df, *cands, required=True):
    for c in cands:
        if c in df.columns: return c
    if required: raise KeyError(f'{cands} 중 없음. 실제 컬럼: {list(df.columns)}')
    return None

for sp in ['train', 'test']:
    p = CLIN_RAW + f'{LABEL}_{sp}.csv.gz'
    print(('OK      ' if os.path.exists(p) else 'MISSING '), p)
for p in [HERG_POOL, CM_OUT + 'crediblemeds_by_key14.csv']:
    print(('OK      ' if os.path.exists(p) else 'MISSING '), p)

OK       C:/Users/iisl/Desktop/0_Molecualar_Informatics/2026_final/2026_new/data/raw/sider_dictrank/Cardiotox (with SIDER inactives)_train.csv.gz
OK       C:/Users/iisl/Desktop/0_Molecualar_Informatics/2026_final/2026_new/data/raw/sider_dictrank/Cardiotox (with SIDER inactives)_test.csv.gz
OK       C:/Users/iisl/Desktop/0_Molecualar_Informatics/2026_final/2026_new/data/processed/herg_karim/herg_karim_pool.csv
OK       C:/Users/iisl/Desktop/0_Molecualar_Informatics/2026_final/2026_new/data/processed/crediblemeds/crediblemeds_by_key14.csv


In [31]:
# ── 셀 2 │ 원본 로드 + 구조 정규화 ──
raw = pd.concat([
    pd.read_csv(CLIN_RAW + f'{LABEL}_{sp}.csv.gz').assign(orig_split=sp)
    for sp in ['train', 'test']], ignore_index=True)

SMI_R = pick(raw, 'Standardized_SMILES', 'smiles', 'SMILES')
INC_R = pick(raw, 'Standardized_InChI', required=False)
N_RAW = len(raw)
print(f'원본 {N_RAW}행  컬럼 {list(raw.columns)}')
print(f'라벨 분포 {raw[LABEL].value_counts().to_dict()}')

def canon(inchi, smi):
    """InChI 우선, 실패 시 SMILES. (canonical SMILES, key14, 원자수) 반환."""
    mol = None
    if isinstance(inchi, str) and inchi.strip():
        try: mol = Chem.MolFromInchi(inchi)
        except Exception: mol = None
    if mol is None:
        mol = Chem.MolFromSmiles(str(smi))
    if mol is None or mol.GetNumAtoms() == 0 or mol.GetNumBonds() == 0:
        return None, None, None
    try: Chem.AssignStereochemistry(mol, cleanIt=True, force=True)
    except Exception: pass
    try: key = Chem.MolToInchiKey(mol).split('-')[0]
    except Exception: key = None
    return Chem.MolToSmiles(mol), key, mol.GetNumAtoms()

out = [canon(r[INC_R] if INC_R else None, r[SMI_R]) for _, r in raw.iterrows()]
raw['canon_smiles'] = [o[0] for o in out]
raw['key14']        = [o[1] for o in out]
raw['n_atoms']      = [o[2] for o in out]

ok = raw['key14'].notna()
N_PARSED = int(ok.sum())
print(f'\nRDKit 실패 {N_RAW - N_PARSED}종 → 사용 가능 {N_PARSED}종')

원본 1253행  컬럼 ['Standardized_SMILES', 'Standardized_InChI', 'Cardiotox (with SIDER inactives)', 'orig_split']
라벨 분포 {1: 742, 0: 511}

RDKit 실패 28종 → 사용 가능 1225종


In [32]:
# ── 셀 3 │ InChIKey14 중복 제거 ──
r = raw[ok].copy()

# 같은 key14 에 라벨이 엇갈리면 그 분자는 통째로 제외
g = r.groupby('key14')[LABEL].nunique()
conflict = set(g[g > 1].index)
print(f'라벨 충돌 key14 {len(conflict)}종 → 제외')

# 접힌 분자(= 같은 key14 가 둘 이상) 기록
dupk = r['key14'].value_counts()
collapsed = int((dupk > 1).sum())
print(f'접힌 그룹 {collapsed}개 (입체이성질체·염 형태 차이)')

pool = (r[~r['key14'].isin(conflict)]
        .drop_duplicates('key14', keep='first')
        .reset_index(drop=True))
pool['y'] = pool[LABEL].astype(int)

def scaffold(smi):
    try: return MurckoScaffold.MurckoScaffoldSmiles(smiles=str(smi), includeChirality=False)
    except Exception: return ''
pool['scaffold'] = pool['canon_smiles'].map(scaffold)

print(f'\n풀 {len(pool)}종  양성 {int((pool["y"]==1).sum())} / '
      f'음성 {int((pool["y"]==0).sum())}  양성률 {pool["y"].mean():.3f}')
print(f'스캐폴드 {pool["scaffold"].nunique()}종')
print(f'\n[Table 1] 원본 {N_RAW} / 파싱 {N_PARSED} / 고유 {len(pool)}')

라벨 충돌 key14 0종 → 제외
접힌 그룹 0개 (입체이성질체·염 형태 차이)

풀 1225종  양성 725 / 음성 500  양성률 0.592
스캐폴드 836종

[Table 1] 원본 1253 / 파싱 1225 / 고유 1225


In [33]:
# ── 셀 4 │ CredibleMeds 주석 머지 ──
cmf = CM_OUT + 'crediblemeds_by_key14.csv'
assert os.path.exists(cmf), f'{cmf} 없음 — CredibleMeds 노트북을 먼저 돌리세요'

cmk = pd.read_csv(cmf)
k2  = pick(cmk, 'key14')
keep = [k2, 'cm_tier'] + [c for c in ['cm_qt', 'cm_tdp', 'cm_ann'] if c in cmk.columns]
cmk = cmk[keep].rename(columns={k2: 'key14'}).drop_duplicates('key14')
print(f'CredibleMeds {len(cmk)}종')

d = pool.merge(cmk, on='key14', how='left')
d['listed'] = d['cm_tier'].notna()

tox = d['y'] == 1
print(f'\n등재 {int(d["listed"].sum())}종  '
      f'(심독성 {int((d["listed"] & tox).sum())}, 비심독성 {int((d["listed"] & ~tox).sum())})')
print('\n등급 분포 (심독성만):')
print(d.loc[d['listed'] & tox, 'cm_tier'].value_counts().to_string())

CredibleMeds 270종

등재 143종  (심독성 133, 비심독성 10)

등급 분포 (심독성만):
cm_tier
Possible       68
Known          36
Conditional    29


In [34]:
# ── 셀 5 │ 엔드포인트 확정 ──
# CredibleMeds 등재 자체를 QT 기전의 지표로 삼는다.
d['endpoint_A'] = (tox &  d['listed']).astype(int)   # 등재 심독성
d['endpoint_B'] = (tox & ~d['listed']).astype(int)   # 미등재 심독성

nA, nB, nN = int(d['endpoint_A'].sum()), int(d['endpoint_B'].sum()), int((~tox).sum())
assert nA + nB == int(tox.sum()), '검산 실패'

def ep(p, n, name):
    p, n = int(p.sum()), int(n.sum())
    return {'Endpoints': name, 'N_pos': p, 'N_neg': n, 'N_total': p + n,
            'Pos_rate': round(p / (p + n), 3)}

A, B = d['endpoint_A'] == 1, d['endpoint_B'] == 1
T2 = pd.DataFrame([
    ep(tox, ~tox, 'Broad (all cardiotoxic vs non-toxic)'),
    ep(A,   ~tox, 'A (listed in CredibleMeds vs non-toxic)'),
    ep(B,   ~tox, 'B (not listed vs non-toxic)'),
    ep(A,    B,   'C (listed vs not listed)'),
])
print(T2.to_string(index=False))
print(f'\n등재된 비심독성 {int(((~tox) & d["listed"]).sum())}종 — 민감도 분석 대상')

                              Endpoints  N_pos  N_neg  N_total  Pos_rate
   Broad (all cardiotoxic vs non-toxic)    725    500     1225     0.592
A (listed in CredibleMeds vs non-toxic)    133    500      633     0.210
            B (not listed vs non-toxic)    592    500     1092     0.542
               C (listed vs not listed)    133    592      725     0.183

등재된 비심독성 10종 — 민감도 분석 대상


In [35]:
# ── 셀 6 │ hERG 중첩 (원고 §3.2) ──
overlap = {}
if os.path.exists(HERG_POOL):
    herg = pd.read_csv(HERG_POOL)
    hk, hy_col = pick(herg, 'key14'), pick(herg, 'y', 'Y', 'label')
    hy = dict(zip(herg[hk], herg[hy_col]))
    cy = dict(zip(d['key14'], d['y']))
    inter = set(hy) & set(cy)
    tab = collections.Counter((int(hy[k]), int(cy[k])) for k in inter)

    n = sum(tab.values())
    conc = tab.get((0,0), 0) + tab.get((1,1), 0)
    print(f'hERG {len(hy)} / 임상 {len(cy)} / 공유 {n}')
    print('\n(hERG, 임상) : 개수')
    for key in [(0,0), (0,1), (1,0), (1,1)]:
        print(' ', key, ':', tab.get(key, 0))
    print(f'\n일치 {conc}/{n} = {conc/n:.1%}')
    print(f'불일치 {n-conc}/{n} = {(n-conc)/n:.1%}')
    print(f'  hERG 음성·임상 양성 : {tab.get((0,1),0)} ({tab.get((0,1),0)/n:.1%})')
    print(f'  hERG 양성·임상 음성 : {tab.get((1,0),0)}')
    overlap = {'n_shared': n, 'concordant': conc,
               'concordance_rate': round(conc/n, 4),
               'herg_neg_clin_pos': tab.get((0,1), 0),
               'herg_pos_clin_neg': tab.get((1,0), 0),
               'crosstab': {str(k): v for k, v in tab.items()}}
else:
    print('hERG 풀 없음 — 건너뜀')

hERG 11037 / 임상 1225 / 공유 209

(hERG, 임상) : 개수
  (0, 0) : 22
  (0, 1) : 98
  (1, 0) : 14
  (1, 1) : 75

일치 97/209 = 46.4%
불일치 112/209 = 53.6%
  hERG 음성·임상 양성 : 98 (46.9%)
  hERG 양성·임상 음성 : 14


In [36]:
# ── 셀 7 │ 저장 ──
d.to_csv(CLIN_OUT + 'clinical_pool_endpoints.csv', index=False, encoding='utf-8-sig')
d.to_csv(CLIN_OUT + 'clinical_pool.csv', index=False, encoding='utf-8-sig')
T2.to_csv(CLIN_OUT + 'table2_endpoints.csv', index=False, encoding='utf-8-sig')

meta = {
    'dataset': 'Cardiotox (with SIDER inactives) - DICTrank + SIDER inactives',
    'source_files': [f'{LABEL}_{sp}.csv.gz' for sp in ['train', 'test']],
    'source_md5': {sp: hashlib.md5(
        open(CLIN_RAW + f'{LABEL}_{sp}.csv.gz', 'rb').read()).hexdigest()
        for sp in ['train', 'test']},
    'n_raw': N_RAW,
    'n_parsed': N_PARSED,
    'n_label_conflict_keys': len(conflict),
    'n_collapsed_groups': collapsed,
    'n_pool': len(d),
    'n_pos': int((d['y'] == 1).sum()), 'n_neg': int((d['y'] == 0).sum()),
    'n_scaffold': int(d['scaffold'].nunique()),
    'dedup_key': 'InChIKey skeleton block (first 14 characters)',
    'dedup_policy': 'drop label-conflicting keys, keep first of remaining duplicates',
    'endpoint_definition': 'A = cardiotoxic AND listed in CredibleMeds; '
                           'B = cardiotoxic AND not listed; negatives = non-cardiotoxic',
    'n_listed': int(d['listed'].sum()),
    'n_listed_toxic': nA,
    'n_listed_nontoxic': int(((~tox) & d['listed']).sum()),
    'overlap_with_herg_karim': overlap,
    'processed_date': datetime.date.today().isoformat(),
}
json.dump(meta, open(CLIN_OUT + 'clinical_provenance.json', 'w', encoding='utf-8'),
          indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in meta.items() if k != 'overlap_with_herg_karim'},
                 indent=2, ensure_ascii=False))

{
  "dataset": "Cardiotox (with SIDER inactives) - DICTrank + SIDER inactives",
  "source_files": [
    "Cardiotox (with SIDER inactives)_train.csv.gz",
    "Cardiotox (with SIDER inactives)_test.csv.gz"
  ],
  "source_md5": {
    "train": "d3abcee7fab138a4c0f18ccae04ad82e",
    "test": "da78e48d421e7ed6034390b567d2005b"
  },
  "n_raw": 1253,
  "n_parsed": 1225,
  "n_label_conflict_keys": 0,
  "n_collapsed_groups": 0,
  "n_pool": 1225,
  "n_pos": 725,
  "n_neg": 500,
  "n_scaffold": 836,
  "dedup_key": "InChIKey skeleton block (first 14 characters)",
  "dedup_policy": "drop label-conflicting keys, keep first of remaining duplicates",
  "endpoint_definition": "A = cardiotoxic AND listed in CredibleMeds; B = cardiotoxic AND not listed; negatives = non-cardiotoxic",
  "n_listed": 143,
  "n_listed_toxic": 133,
  "n_listed_nontoxic": 10,
  "processed_date": "2026-11-13"
}


In [18]:
import json, datetime

json.dump(split_idx,
          open(CLIN_OUT + 'clinical_orig_splits.json', 'w', encoding='utf-8'))

meta = {
    'dataset': 'Cardiotox (with SIDER inactives) - Srijit et al., DICTrank + SIDER inactives',
    'source_files': [f'{LABEL}_{sp}.csv.gz' for sp in ['train', 'test']],
    # ... (나머지 그대로)
    'processed_date': datetime.date.today().isoformat(),
}
json.dump(meta,
          open(CLIN_OUT + 'clinical_provenance.json', 'w', encoding='utf-8'),
          indent=2, ensure_ascii=False)
print(json.dumps(meta, indent=2, ensure_ascii=False))

{
  "dataset": "Cardiotox (with SIDER inactives) - Srijit et al., DICTrank + SIDER inactives",
  "source_files": [
    "Cardiotox (with SIDER inactives)_train.csv.gz",
    "Cardiotox (with SIDER inactives)_test.csv.gz"
  ],
  "processed_date": "2026-11-12"
}


In [37]:
# ── 셀 8 │ 원고용 표 출력 ──
print('[Table 1 행 — 탭 구분]')
print('Dataset\tRole\tRaw\tUnique by InChIKey14\tToxic\tNon-toxic')
print(f'Srijit et al.\tTarget (clinical)\t{N_RAW}\t{len(d)}\t'
      f'{int((d["y"]==1).sum())}\t{int((d["y"]==0).sum())}')

print('\n[Table 3 엔드포인트 — 탭 구분]')
print('\t'.join(T2.columns))
for _, r in T2.iterrows():
    print('\t'.join(str(v) for v in r.values))

[Table 1 행 — 탭 구분]
Dataset	Role	Raw	Unique by InChIKey14	Toxic	Non-toxic
Srijit et al.	Target (clinical)	1253	1225	725	500

[Table 3 엔드포인트 — 탭 구분]
Endpoints	N_pos	N_neg	N_total	Pos_rate
Broad (all cardiotoxic vs non-toxic)	725	500	1225	0.592
A (listed in CredibleMeds vs non-toxic)	133	500	633	0.21
B (not listed vs non-toxic)	592	500	1092	0.542
C (listed vs not listed)	133	592	725	0.183
